# 03 — Gold: bid performanceBusiness-facing aggregates. Two rules shape this layer:**Every rate is reported twice** — including and excluding bulk-loadedrecords. A single number here would be misleading, and which one is "correct"depends on the question being asked.**Win rate is reported by count and by value.** The two differ by roughly fivepoints, and only the value-weighted figure reflects commercial reality.Sales-cycle length is deliberately absent. More than half the closuretimestamps were written in bulk-closing sessions, seconds apart, months afterthe fact — any duration derived from them would be fiction.

In [0]:
from pyspark.sql import functions as Fspark.sql("CREATE SCHEMA IF NOT EXISTS gold.bid")bids = spark.table("silver.bid.bids_clean")clients = spark.table("silver.bid.clients_clean")fact = (    bids.join(F.broadcast(clients), "client_id", "left")        .filter(F.col("outcome").isNotNull())      # closed bids only)

## Core metrics`win_rate_by_count` treats a R$20k bid and a R$2M bid identically.`win_rate_by_value` does not. Publishing only the first overstatesperformance.

In [0]:
def performance(df, *dims):    return (        df.groupBy(*dims)          .agg(              F.count("*").alias("bids_closed"),              F.sum("outcome").alias("bids_won"),              F.round(F.avg("outcome") * 100, 1).alias("win_rate_by_count"),              F.round(                  F.sum(F.when(F.col("outcome") == 1, F.col("contract_value_brl")).otherwise(0))                  / F.sum("contract_value_brl") * 100, 1              ).alias("win_rate_by_value"),              F.round(F.sum("contract_value_brl"), 2).alias("value_bid_brl"),          )    )overall = performance(fact, F.lit(True).alias("_all")).drop("_all")by_channel = performance(fact, "is_bulk_load")overall.write.format("delta").mode("overwrite").saveAsTable("gold.bid.performance_overall")by_channel.write.format("delta").mode("overwrite").saveAsTable("gold.bid.performance_by_channel")display(by_channel)

## Segmentation, with the confound isolatedEach dimension is reported on all closed bids and again on organic bids only.The gap between the two columns is the size of the migration artefact — andfor at least one executive it is the difference between "underperforming" and"above average".

In [0]:
organic = fact.filter(~F.col("is_bulk_load"))for dim in ["segment", "state", "account_executive"]:    combined = (        performance(fact, dim)        .select(dim, "bids_closed", F.col("win_rate_by_count").alias("wr_all"))        .join(            performance(organic, dim)              .select(dim,                      F.col("bids_closed").alias("bids_organic"),                      F.col("win_rate_by_count").alias("wr_organic")),            dim, "left",        )        .withColumn("artefact_gap", F.round(F.col("wr_organic") - F.col("wr_all"), 1))        .orderBy(F.col("bids_closed").desc())    )    combined.write.format("delta").mode("overwrite").saveAsTable(f"gold.bid.performance_by_{dim}")    display(combined)

## Contract value effectQuartiles are computed across closed bids. If the win rate falls as valuerises, count-based reporting is systematically flattering.

In [0]:
value_bands = (    fact    .withColumn("value_quartile", F.ntile(4).over(        __import__("pyspark").sql.Window.orderBy("contract_value_brl"))))by_value = (    performance(value_bands, "value_quartile")    .orderBy("value_quartile"))by_value.write.format("delta").mode("overwrite").saveAsTable("gold.bid.performance_by_value_band")display(by_value)

## Loss reasons, and how little of the picture they coverThe coverage figure is the point of this table. A reason distribution builton a single-digit share of losses is a signal, not a population estimate, andthe two must be published together.

In [0]:
losses = bids.filter(F.col("outcome") == 0)coverage = losses.agg(    F.count("*").alias("losses_total"),    F.sum(F.col("has_loss_reason").cast("int")).alias("losses_with_reason"),    F.round(F.avg(F.col("has_loss_reason").cast("int")) * 100, 1).alias("coverage_pct"),    F.sum(F.col("competitor_is_placeholder").cast("int")).alias("placeholder_attributed"),)reasons = (    losses.filter(F.col("has_loss_reason"))          .groupBy("loss_reason")          .agg(F.count("*").alias("losses"))          .withColumn("share_pct", F.round(              F.col("losses") / F.sum("losses").over(                  __import__("pyspark").sql.Window.partitionBy()) * 100, 1))          .orderBy(F.col("losses").desc()))coverage.write.format("delta").mode("overwrite").saveAsTable("gold.bid.loss_reason_coverage")reasons.write.format("delta").mode("overwrite").saveAsTable("gold.bid.loss_reasons")display(coverage)display(reasons)

## Open pipelineBids with a NULL outcome. Reported separately so they never silently join theloss column, which would understate the win rate by roughly a third.

In [0]:
pipeline = (    bids.filter(F.col("outcome").isNull())        .join(F.broadcast(clients), "client_id", "left")        .groupBy("segment")        .agg(            F.count("*").alias("bids_open"),            F.round(F.sum("contract_value_brl"), 2).alias("value_open_brl"),        )        .orderBy(F.col("value_open_brl").desc()))pipeline.write.format("delta").mode("overwrite").saveAsTable("gold.bid.open_pipeline")display(pipeline)